## Execution

In [46]:
LOAD_MODEL = True
CREATE_DATASET = False
TRAIN_MODEL = False
EVAL_MODEL = True
SAVE_MODEL = True

## Autenticação

In [ ]:
from google.colab import auth
auth.authenticate_user()

## Utils

In [ ]:
!pip3 install torchmetrics

In [ ]:
import torch
import numpy as np
from torchaudio import functional as AF
import torchaudio
import torch.nn as nn
import torch.optim as optim
from torchmetrics.audio import ScaleInvariantSignalDistortionRatio, SignalDistortionRatio
import torchmetrics
import matplotlib.pyplot as plt
import os

def downmix_to_mono(waveform):
  return torch.mean(waveform, dim=0, keepdim=True)

# No longer be used this function
def resample(waveform, rate_of_sample, new_rate_sample):
    waveform = AF.resample(
        waveform,
        orig_freq=rate_of_sample,
        new_freq=new_rate_sample
    )
    return waveform, new_rate_sample

def audio_to_waveform(path):
    data_waveform, rate_of_sample = torchaudio.load(path)
    return data_waveform, rate_of_sample

class TransformSpec:
    def __init__(self):
        self.n_fft = 2048
        self.win_length = 2048
        self.hop_length = self.n_fft // 4
        self.window_fn=torch.hann_window
        self.power = None
        self.center = True
        self.pad_mode = "reflect"

    def transform_in_spectogram(self, data_waveform):
        audio_spectogram = torchaudio.transforms.Spectrogram(
            n_fft=self.n_fft,
            hop_length=self.hop_length,
            window_fn=self.window_fn,
            power=self.power,
            center=self.center,
            pad_mode=self.pad_mode
        )
        audio_spectogram = audio_spectogram(data_waveform)
        magnitude = audio_spectogram.abs()
        phase = audio_spectogram.angle()
        return magnitude, phase

    def transform_in_waveform(self, data_spectogram):
        waveform = torch.istft(
            data_spectogram,
            n_fft=self.n_fft,
            hop_length=self.hop_length,
            win_length=self.win_length,
            window=torch.hann_window(self.win_length)
        )
        return waveform

def avaliation_model(model, learning_rate):
    loss_fn = nn.L1Loss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    return loss_fn, optimizer

def save_checkpoint(state, filename):
    print("=> Saving checkpoint")
    torch.save(state, filename)

def load_checkpoint(checkpoint, model):
    if os.path.isfile(checkpoint):
        checkpoint = torch.load(checkpoint)
        print("=> Loading checkpoint")
        model.load_state_dict(checkpoint["state_dict"])
    else:
        print("=> No checkpoint")

#Verificar se essa função está correta
def torch_to_tensor(features, targets, phase=None, waveform=None, purpose="train", device="cpu"):
    # Transformar [1, 513, 21062, 4] em [1, 4, 513, 21062] e passar para pytorch
    features = torch.tensor(features.numpy()).permute(0, 2, 1, 3).to(device)
    targets = torch.tensor(targets.numpy()).permute(0, 3, 1, 2).float().to(device)
    # torch.Size([8, 4, 88200])
    # torch.Size([8, 4, 169, 1025])
    if purpose == "eval":
        waveform = torch.tensor(waveform.numpy()).permute(0, 2, 1).to(device)
        waveform = waveform.squeeze()
        phase = torch.tensor(phase.numpy()).to(device)
        phase = phase.squeeze()
        return features, targets, phase, waveform
    return features, targets

def train_model(features, targets, model, loss_fn, optimizer, epoch, batch_idx, loss_list, max_epochs, max_batchs, device):
    features, targets = torch_to_tensor(features, targets, device=device)
    with torch.cuda.amp.autocast():
        predictions = model(features)
        loss = loss_fn(predictions, targets)

    loss.backward()
    optimizer.step()

    print(f"Epoch: {epoch+1}/{max_epochs} | Batch: {batch_idx+1}/{max_batchs} | Loss: {loss.item()}")
    loss_list.append(loss.item())

    return loss, predictions, features, targets
    # optimizer.zero_grad()
    # scaler.scale(loss).backward()
    # scaler.step(optimizer)
    # scaler.update()

    # # update tqdm loop
    # loop.set_postfix(loss=loss.item())


def evaluation_model(model, ds_test, max_batchs, device):
    model.eval()
    NUM_CLASSES = 4
    si_sdr = ScaleInvariantSignalDistortionRatio()
    si_sdr_metrics = []
    transform_spec = TransformSpec()
    for batch_idx, (features, targets, phase) in enumerate(ds_test):
        si_sdr_metric = []
        features, targets, phase = torch_to_tensor(features, targets, phase, "eval", device=device)
        predict = model(features)
        with torch.no_grad():
            for ch, data in enumerate(predict[0]):
                reconstructed_predict = data * torch.exp(1j * phase)
                reconstructed_targets = targets[0][ch] * torch.exp(1j * phase)
                waveform_predict = transform_spec.transform_in_waveform(reconstructed_predict)
                waveform_targets = transform_spec.transform_in_waveform(reconstructed_targets)
                si_sdr_metric.append(si_sdr(waveform_predict, waveform_targets))
            print(f"Batch: {batch_idx+1}/{max_batchs} | SI-SDR: {sum(si_sdr_metric) / len(si_sdr_metric)}")
            si_sdr_metrics.append(sum(si_sdr_metric) / len(si_sdr_metric))
    return si_sdr_metrics

def plot_data(metric, i, purpose):
    print("=> saving results")
    plt.figure()
    plt.plot(metric)
    plt.xlabel("Iterations")
    metric_type = "loss" if purpose == "train" else "si_sdr"
    plt.ylabel(metric_type)
    plt.title(f"Training metric_type Evolution")
    plt.savefig(f"results_img/{metric_type}_{i}.png")

## Dataset

In [ ]:
import os
import tensorflow as tf
import numpy as np
# from .Utils import TransformSpec, \
#                     audio_to_waveform, \
#                     downmix_to_mono, \
#                     resample
import torch
import os

transform_spec = TransformSpec()

def _bytes_feature(value):
  if isinstance(value, type(tf.constant(0))):
    value = value.numpy()
  elif isinstance(value, str):
    value = value.encode("utf-8")
  return tf.train.Feature(bytes_list=tf.train.BytesList(value=[value]))

def load_data_path(dataset_path):
    subsets = []
    for subset in ["train", "test"]:
        subset_path = os.path.join(dataset_path, subset)
        samples = []
        for track in os.listdir(subset_path):
            track_path = os.path.join(subset_path, track)
            if track == ".ipynb_checkpoints":
              continue
            print(track_path)
            if not os.path.isdir(track_path):
                continue
            paths = {
                0: os.path.join(track_path, "mixture.wav"),
                1: os.path.join(track_path, "vocals.wav"),
                2: os.path.join(track_path, "bass.wav"),
                3: os.path.join(track_path, "drums.wav"),
                4: os.path.join(track_path, "other.wav")
            }
            samples.append(paths)
        subsets.append(samples)
    return subsets

def save_data(data, writer, type, new_rate_sample):
    tensors = {}
    chunks = {}
    phase = None
    segment=2 #5 segundos cada chunk

    for index, item in data.items():
        data_waveform, rate_of_sample = audio_to_waveform(item)
        data_waveform = downmix_to_mono(data_waveform)
        # data_waveform, rate_of_sample = resample(data_waveform, rate_of_sample, new_rate_sample)
        tensors[index] = data_waveform
    mix = tensors[0][None]
    chunk_len = int(rate_of_sample * segment)
    length = mix.shape[-1]
    end = chunk_len
    start = 0
    threshold = 0.001
    while start < length:
        chunk_mix = mix[:, :, start:end]

        rms = torch.sqrt(torch.mean(chunk_mix**2))
        if rms < threshold:
            start += chunk_len
            end = start + chunk_len
            continue

        for i, source in tensors.items():
            source_separated = source[None]  # [1, C, T]
            chunk = source_separated[:, :, start:end]

            if chunk.shape[-1] < chunk_len:
                pad_size = chunk_len - chunk.shape[-1]
                chunk = torch.nn.functional.pad(chunk, (0, pad_size))

            chunk = chunk.squeeze(0)
            chunk = chunk.to(torch.float32).cpu().numpy()
            serialized_chunk = tf.io.serialize_tensor(chunk)

            chunks[i] = serialized_chunk


        features = {
            "mix": _bytes_feature(chunks[0]),
            "vocals": _bytes_feature(chunks[1]),
            "bass": _bytes_feature(chunks[2]),
            "drums": _bytes_feature(chunks[3]),
            "others": _bytes_feature(chunks[4]),
        }

        row = tf.train.Example(features=tf.train.Features(feature=features))
        writer.write(row.SerializeToString())

        start += chunk_len
        end = start + chunk_len

def Get_dataset(dataset_path, new_rate_sample):
    print("=> Create TFRecords")
    os.system('cls' if os.name == 'nt' else 'clear')
    tf_train, tf_test = load_data_path(dataset_path)

    df = {
        "train": tf_train,
        "test": tf_test
    }

    for type, tf_type in df.items():
        PATH_DFRECORDS = f"./TFRecords/{type}"
        if not os.path.exists(PATH_DFRECORDS):
            os.makedirs(PATH_DFRECORDS)

        for shard in range(len(tf_type)):
            output_filename = os.path.join(PATH_DFRECORDS,'{}_{:03d}-of-{:03d}.tfrecord'.format("audios_sources", shard+1, len(tf_type)))
            data_shard = tf_type[shard:shard+1]
            for data in data_shard:
                print(f"{shard+1} of {len(tf_type)} - {type}")
                options = tf.io.TFRecordOptions(compression_type="GZIP")
                with tf.io.TFRecordWriter(output_filename, options=options) as writer:
                        save_data(data, writer, type, new_rate_sample)

## Model

In [ ]:
import torch
import torch.nn as nn
import torchvision.transforms.functional as TF

class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(DoubleConv, self).__init__()
        self.kernel_size = 3
        self.stride = 1
        self.padding = 1
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=self.kernel_size, stride=self.stride, padding=self.padding, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),

            nn.Conv2d(out_channels, out_channels, kernel_size=self.kernel_size, stride=self.stride, padding=self.padding, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.conv(x)

class UpSampling(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(UpSampling, self).__init__()
        self.kernel_size = 5
        self.stride = 2
        self.padding = 2
        self.dropout = 0.4
        self.up = nn.Sequential (
            nn.ConvTranspose2d(in_channels, out_channels, kernel_size=self.kernel_size, stride=self.stride, padding=self.padding, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Dropout2d(p=self.dropout)
        )
    def forward(self, x):
        return self.up(x)

class DoubleDeconv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(DoubleDeconv, self).__init__()
        self.kernel_size = 3
        self.stride = 2
        self.padding = 1
        self.deconv = nn.Sequential(
            nn.ConvTranspose2d(in_channels, out_channels, kernel_size=self.kernel_size, padding=self.padding, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),

            nn.ConvTranspose2d(out_channels, out_channels, kernel_size=self.kernel_size, padding=self.padding, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.deconv(x)


class UNet(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(UNet, self).__init__()
        self.channels = in_channels
        self.outs = [16, 32, 64, 128, 256, 512]
        self.ups = nn.ModuleList()
        self.downs = nn.ModuleList()
        self.deconv = nn.ModuleList()
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        # Down part of UNET (Encoder)
        for i, out in enumerate(self.outs):
            if i+1 == len(self.outs):
                self.bottleneck = DoubleConv(self.channels, self.outs[i])
                break

            self.downs.append(DoubleConv(self.channels, self.outs[i]))
            self.channels = out

        # Up part of UNET (Decoder)
        for i,out in enumerate(reversed(self.outs[:-1])):
            self.ups.append(UpSampling(out*2, out))
            self.deconv.append(DoubleDeconv(out*2, out))

            #final_conv
            if i+1 == len(self.outs[:-1]):
                self.final_conv = nn.Sequential(
                    nn.Conv2d(out, out_channels, kernel_size=1),
                    nn.Softplus()
                )

    def forward(self, x):
        skip_connections = []

        for down in self.downs:
            x = down(x)
            print(1)
            skip_connections.append(x)
            x = self.pool(x)

        x = self.bottleneck(x)
        skip_connections = skip_connections[::-1]

        for i in range(len(self.ups)):
            x = self.ups[i](x)
            skip_connection = skip_connections[i]
            if x.shape != skip_connection.shape:
                x = TF.resize(x, size=skip_connection.shape[2:])
            concat_skip = torch.cat((skip_connection, x), dim=1)
            x = self.deconv[i](concat_skip)

        return self.final_conv(x)

## Transform and validate data

In [ ]:
import tensorflow as tf

def spectrogram_to_waveform(magnitude, phase):
    stft_complex = tf.cast(magnitude, tf.complex64) * tf.exp(1j * tf.cast(phase, tf.complex64))

    waveform = tf.signal.inverse_stft(
        stft_complex,
        frame_length=2048,
        frame_step=2048 // 4,
        window_fn=tf.signal.inverse_stft_window_fn(
            2048 // 4,
            forward_window_fn=tf.signal.hann_window
        )
    )
    return waveform

def parse_data(row_data, purpose):
    base_features = ["mix", "vocals", "bass", "drums", "others"]
    # if purpose == "eval":
    #     base_features.append("original_phase")

    feature_description = {
        key: tf.io.FixedLenFeature([], tf.string)
        for key in base_features
    }

    parsed = tf.io.parse_single_example(row_data, feature_description)

    feature_deserialized = []
    for key in base_features:
        tensor = tf.io.parse_tensor(parsed[key], out_type=tf.float32)
        tensor = tf.squeeze(tensor)
        feature_deserialized.append(tensor)

    return feature_deserialized

def split_data(*row_data, purpose):
    X = tf.expand_dims(row_data[0], axis=1)
    y = tf.stack([row_data[1], row_data[2], row_data[3], row_data[4]], axis=-1)
    phase = tf.expand_dims(row_data[5], axis=1) if purpose == "eval" else None
    return X, y, phase

def set_original_ds(*row_data):
  return tf.stack([row_data[1], row_data[2], row_data[3], row_data[4]], axis=-1)

def spectrogram_parse(*row_data, purpose):
    row_data_spectrogram = []
    for i,row in enumerate(row_data):
      stft = tf.signal.stft(
          row,
          frame_length=2048,
          frame_step=2048 // 4,
          window_fn=tf.signal.hann_window
      )
      magnitude = tf.abs(stft)
      row_data_spectrogram.append(magnitude)

      if purpose == "eval" and i == len(row_data)-1:
        stft = tf.signal.stft(
            row_data[0],
            frame_length=2048,
            frame_step=2048 // 4,
            window_fn=tf.signal.hann_window
        )
        phase = tf.math.angle(stft)
        row_data_spectrogram.append(phase)

    row_data_spectrogram = tuple(row_data_spectrogram)
    return row_data_spectrogram

def get_data(path, batch_size, purpose):
    print(f"=> Get {purpose} data")
    ds = tf.data.TFRecordDataset.list_files(path, shuffle=False).interleave(
        lambda x: tf.data.TFRecordDataset(x, compression_type="GZIP"),
        num_parallel_calls=tf.data.AUTOTUNE,
        cycle_length=tf.data.AUTOTUNE
    )

    ds = ds.map(lambda x: parse_data(x, purpose), num_parallel_calls=tf.data.AUTOTUNE)
    if purpose == "eval":
      ds_sources_waveform = ds.map(lambda *x: set_original_ds(*x), num_parallel_calls=tf.data.AUTOTUNE)

    ds = ds.map(lambda *x: spectrogram_parse(*x, purpose=purpose), num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.map(lambda *x: split_data(*x, purpose=purpose), num_parallel_calls=tf.data.AUTOTUNE)

    ds = ds.batch(batch_size)
    ds = ds.prefetch(buffer_size=tf.data.AUTOTUNE)

    if purpose == "eval":
      ds_sources_waveform = ds_sources_waveform.batch(batch_size)
      ds_sources_waveform = ds_sources_waveform.prefetch(tf.data.AUTOTUNE)
      return ds, ds_sources_waveform
    return ds

## Train

In [ ]:
import tensorflow as tf

# https://apxml.com/courses/getting-started-with-tensorflow/chapter-5-data-input-pipelines-tfdata/working-tfrecord-files
# https://www.tensorflow.org/api_docs/python/tf/data/experimental/parallel_interleave

def train(ds_train, model, num_epochs, learning_rate, model_saved_location, device, count):
    print("=> Training the model")
    loss_list = []
    loss_fn, optimizer = avaliation_model(model, learning_rate)
    for epoch in range(num_epochs):
        for batch_idx, (features, targets, _) in enumerate(ds_train):
            train_model(features, targets, model, loss_fn, optimizer, epoch, batch_idx,loss_list, num_epochs, count, device)
        checkpoint = {
            "state_dict": model.state_dict(),
            "optimizer":optimizer.state_dict(),
        }
        save_checkpoint(checkpoint, model_saved_location)

    return loss_list

# https://www.tensorflow.org/tutorials/load_data/tfrecord?hl=pt-br#reading_a_tfrecord_file

## Evaluation

In [ ]:
from torchmetrics.audio import ScaleInvariantSignalDistortionRatio, SignalDistortionRatio

def evaluation(ds_test, ds_waveform, model, device, count):
  print("=> Evaluating the model")
  model.eval()
  si_sdr = ScaleInvariantSignalDistortionRatio().to(device)
  si_sdr_metrics = []
  # sdr = SignalDistortionRatio()
  # sdr_metrics = []
  for batch_idx, ((features_tf, targets_tf, phase_tf), waveform_batch_tf) in enumerate(zip(ds_test, ds_waveform)):
    si_sdr_metric = []
    sdr_metric = []
    features_tf, targets_tf, phase_tf, waveform_batch_tf = torch_to_tensor(features_tf, targets_tf, phase_tf, waveform_batch_tf, "eval", device=device)
    with torch.no_grad():
      predict = model(features_tf)
      for i,batch in enumerate(predict):
        reconstructed_predict = spectrogram_to_waveform(batch.cpu().numpy(), phase_tf[i].cpu().numpy())
        predict = torch.from_numpy(reconstructed_predict.numpy()).to(device).float()

        target = waveform_batch_tf[i].float()

        min_len = min(predict.shape[-1], target.shape[-1])
        predict = predict[..., :min_len]
        target = target[..., :min_len]

        si_sdr_metric.append(si_sdr(predict, target))
      print(f"Batch: {batch_idx+1}/{count} | SI-SDR: {sum(si_sdr_metric) / len(si_sdr_metric)}")
      si_sdr_metrics.append(sum(si_sdr_metric) / len(si_sdr_metric))
  return si_sdr_metrics

## Pipeline

In [ ]:
!mkdir model
!mkdir results_img/

In [ ]:
import json

def WriteFile(attempt_number=0):
    nome_arquivo = "attempt.json"
    attempt = {
        "attempt": attempt_number + 1
    }
    try:
        arquivo = open(nome_arquivo, 'w', encoding='utf-8')
        json.dump(attempt, arquivo, indent=4, ensure_ascii=False)

    except FileNotFoundError:
        arquivo = open(nome_arquivo, 'w+')
        WriteFile()
    #faca o que quiser
    arquivo.close()

def ReadFile():
    nome_arquivo = "attempt.json"
    attempt = {
        "attempt": 1
    }
    try:
        arquivo = open(nome_arquivo, 'r', encoding='utf-8')
        attempt = json.load(arquivo)
        return attempt

    except FileNotFoundError:
        arquivo = open(nome_arquivo, 'w+')
        ReadFile()
    arquivo.close()
    return

WriteFile()

In [ ]:
import subprocess

def SaveModel():
    filename = f"{LOCATION_FILE_MODEL}_{ATTEMPT}{FORMAT_FILE_MODEL}"
    copy_command = subprocess.run(["gcloud", "storage", "cp", filename, SAVE_GS_FILE_MODEL], capture_output=True, text=True)
    print(copy_command.stdout)

def LoadModel():
    filename = f"{LOCATION_FILE_MODEL}_{ATTEMPT}{FORMAT_FILE_MODEL}"
    copy_command = subprocess.run(["gcloud", "storage", "cp", SAVE_GS_FILE_MODEL+"my_checkpoint_1.pth.tar", filename], capture_output=True, text=True)
    print(copy_command.stdout)

In [ ]:
import torch
import argparse
import multiprocessing

num_nucleos = multiprocessing.cpu_count()

LEARNING_RATE = 1e-6
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NUM_EPOCHS = 1
TF_LOCATION_TRAIN="gs://tf_records_file_source_audio_separation/train/*.tfrecord"
TF_LOCATION_TEST="gs://tf_records_file_source_audio_separation/test/*.tfrecord"
SAVE_GS_FILE_MODEL="gs://model_unet_for_source_audio_separation/"
LOCATION_FILE_MODEL = "./model/my_checkpoint"
FORMAT_FILE_MODEL = ".pth.tar"
IN_CHANNELS=1
OUT_CHANNELS=4
BATCH_SIZE=8
ATTEMPT=int(ReadFile()["attempt"])

In [ ]:
# Get_dataset(AUDIO_LOCATION, NEW_RATE_SAMPLE)

In [ ]:
model = UNet(in_channels=IN_CHANNELS, out_channels=OUT_CHANNELS).to(DEVICE)

In [ ]:
if LOAD_MODEL:
  LoadModel()
  load_checkpoint(f"{LOCATION_FILE_MODEL}_{ATTEMPT}{FORMAT_FILE_MODEL}", model)

In [ ]:
if TRAIN_MODEL:
  count_train = 0
  ds_train = get_data(TF_LOCATION_TRAIN, BATCH_SIZE, "train")
  for _ in ds_train: #armazenar quantos dados há dentro do dataset
      count_train += 1

In [ ]:
if TRAIN_MODEL:
  loss_list = train(ds_train, model, NUM_EPOCHS, LEARNING_RATE, f"{LOCATION_FILE_MODEL}_{ATTEMPT}{FORMAT_FILE_MODEL}", DEVICE, count_train)
  plot_data(loss_list, ATTEMPT, "train")

In [ ]:
if EVAL_MODEL:
  ds_eval_spec, ds_eval_wave = get_data(TF_LOCATION_TEST, BATCH_SIZE, "eval")
  count_test = 0
  for _ in ds_eval_spec: #armazenar quantos dados há dentro do dataset
      count_test += 1

=> Get eval data


In [ ]:
if EVAL_MODEL:
  evaluation(ds_eval_spec, ds_eval_wave, model, DEVICE, count_test)

In [ ]:
if SAVE_MODEL:
  SaveModel()

In [ ]:
WriteFile(ATTEMPT)